# YoungNAKO · Machine Learning · Hands-on 3
#### Thomas Küstner, University Hospital of Tübingen
## Does it actually work? Evaluation and pitfalls

**Thursday 05.11.2026, Block 3 (≈ 30 min)** · Thomas Küstner, MIDAS.lab, University Hospital Tübingen

A model that scores AUC = 0.99 can still be useless. We break our own models on purpose:

| Part | Pitfall | ≈ time |
|---|---|---|
| 1 | **Data leakage**: slice-level vs patient-level split | 5 min |
| 2 | **Domain shift**: new contrast, noise, bias field | 7 min |
| 3 | **Calibration**: does p = 0.9 mean 90 %? | 5 min |
| 4 | **Segmentation metrics**: Dice vs Hausdorff | 5 min |
| 5 | **Shortcut learning**: the "Clever Hans" model → bridge to *Explainable AI* tomorrow | 8 min |

> ⚙️ `Runtime → Change runtime type → T4 GPU` (each training run then takes a few seconds).

In [ ]:
#@title 📦 Setup { display-mode: "form" }
# Install the requirements into the running kernel: requirements.txt of the repo if it is next to
# this notebook (local / cloned repo), otherwise (Colab opened via link) this notebook's packages.
import os
_req = next((p for p in ("requirements.txt", "../requirements.txt") if os.path.isfile(p)), None)
get_ipython().run_line_magic("pip", f"install -q -r {_req}" if _req else
                             "install -q numpy pandas matplotlib scipy scikit-learn torch pillow pydicom SimpleITK")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("Running on:", DEVICE)

In [ ]:
#@title 🔧 Helper functions: download & load the CHAOS MRI data (just run this cell) { display-mode: "form" }
# CHAOS challenge data (Kavur et al., Med Image Anal 2021), Zenodo record 3431873, licence CC BY-NC-SA 4.0.
# We use the 20 MR training cases, T1-DUAL sequence: in-phase (IP) and opposed-phase (OP) images
# plus expert masks of liver, right kidney, left kidney and spleen.
import os, glob, zipfile, urllib.request, shutil
import numpy as np

CHAOS_URL = "https://zenodo.org/records/3431873/files/CHAOS_Train_Sets.zip?download=1"
# Colab: /content/chaos · locally: ~/yn_data/chaos (shared by all notebooks, outside Dropbox/git); override with YN_CHAOS_ROOT
CHAOS_DIR = os.environ.get("YN_CHAOS_ROOT", "/content/chaos" if os.path.isdir("/content") else os.path.expanduser("~/yn_data/chaos"))
LABELS = {1: "liver", 2: "kidney_right", 3: "kidney_left", 4: "spleen"}


def download_chaos(dest=CHAOS_DIR):
    """Download (≈ 0.9 GB) and unpack only the MR part of the CHAOS training set."""
    if glob.glob(os.path.join(dest, "**", "T1DUAL"), recursive=True):
        print("CHAOS data already available in", dest)
        return dest
    os.makedirs(dest, exist_ok=True)
    zpath = os.path.join(dest, "CHAOS_Train_Sets.zip")
    if not os.path.exists(zpath):
        print("Downloading CHAOS training set from Zenodo (≈ 0.9 GB, 1-3 min on Colab) ...")
        with urllib.request.urlopen(CHAOS_URL) as r, open(zpath, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 22)
    with zipfile.ZipFile(zpath) as z:
        members = [m for m in z.namelist() if "/MR/" in m.replace("\\", "/") and "T1DUAL" in m]
        z.extractall(dest, members)
    os.remove(zpath)
    print("Done:", len(glob.glob(os.path.join(dest, "**", "T1DUAL"), recursive=True)), "MR cases")
    return dest


def _png_to_labels(png):
    """CHAOS ground-truth PNGs encode organs as grey values: liver 63, R kidney 126, L kidney 189, spleen 252."""
    lab = np.zeros(png.shape, np.uint8)
    lab[(png >= 55) & (png <= 70)] = 1
    lab[(png >= 110) & (png <= 135)] = 2
    lab[(png >= 175) & (png <= 200)] = 3
    lab[(png >= 240)] = 4
    return lab


def load_case(case_dir):
    """Load one CHAOS T1-DUAL case.
    Returns dict with 'ip', 'op' (float32, z×y×x), 'mask' (uint8, z×y×x),
    'spacing' (x, y, z in mm), 'origin', 'direction' (for building a 3D image), 'id'."""
    import pydicom
    from PIL import Image
    t1 = os.path.join(case_dir, "T1DUAL")
    ip_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "InPhase", "*.dcm")))
    op_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "OutPhase", "*.dcm")))
    gt_files = sorted(glob.glob(os.path.join(t1, "Ground", "*.png")))
    n = min(len(ip_files), len(op_files), len(gt_files))
    if not (len(ip_files) == len(op_files) == len(gt_files)):
        print(f"  note: case {os.path.basename(case_dir)} has {len(ip_files)}/{len(op_files)}/{len(gt_files)} IP/OP/GT slices, using {n}")
    ip_ds = [pydicom.dcmread(f) for f in ip_files[:n]]
    op_ds = [pydicom.dcmread(f) for f in op_files[:n]]
    gt = [_png_to_labels(np.array(Image.open(f).convert("L"))) for f in gt_files[:n]]
    # geometry from DICOM header -> sort slices along the slice normal
    iop = np.array(ip_ds[0].ImageOrientationPatient, float)
    r, c = iop[:3], iop[3:]
    nrm = np.cross(r, c)
    pos = np.array([np.dot(np.array(d.ImagePositionPatient, float), nrm) for d in ip_ds])
    order = np.argsort(pos)
    ip = np.stack([ip_ds[i].pixel_array.astype(np.float32) for i in order])
    op = np.stack([op_ds[i].pixel_array.astype(np.float32) for i in order])
    mask = np.stack([gt[i] for i in order])
    dz = float(np.median(np.diff(pos[order]))) if n > 1 else float(ip_ds[0].SliceThickness)
    ps = [float(v) for v in ip_ds[0].PixelSpacing]  # [row spacing, column spacing]
    direction = (r[0], c[0], nrm[0], r[1], c[1], nrm[1], r[2], c[2], nrm[2])
    origin = tuple(float(v) for v in ip_ds[order[0]].ImagePositionPatient)
    return dict(ip=ip, op=op, mask=mask, spacing=(ps[1], ps[0], abs(dz)), origin=origin,
                direction=direction, id=os.path.basename(case_dir))


def load_all_cases(root=CHAOS_DIR):
    case_dirs = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "MR", "*", "T1DUAL"), recursive=True)},
                       key=lambda s: int(os.path.basename(s)) if os.path.basename(s).isdigit() else s)
    cases = [load_case(d) for d in case_dirs]
    print(f"Loaded {len(cases)} cases, {sum(c['ip'].shape[0] for c in cases)} slices per contrast")
    return cases


def to_sitk(vol, case):
    """Wrap a z×y×x numpy array into a SimpleITK image with the case geometry."""
    import SimpleITK as sitk
    img = sitk.GetImageFromArray(vol)
    img.SetSpacing(case["spacing"]); img.SetOrigin(case["origin"]); img.SetDirection(case["direction"])
    return img


def _norm(vol):
    lo, hi = np.percentile(vol, 1), np.percentile(vol, 99.5)
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)


def make_slice_dataset(cases, size=128, min_kidney_px=50):
    """Turn the 3D cases into a 2D slice dataset.
    X_ip, X_op : (N, size, size) images scaled to [0, 1] (per volume)
    y_kidney   : 1 if at least one kidney is visible in the slice
    z_rel      : relative slice position in the volume (0 = lowest slice, 1 = highest)
    patient    : case id (for patient-level splitting!)
    organs     : list of visible organs per slice (used for captions)"""
    import torch, torch.nn.functional as F
    X_ip, X_op, M, yk, zr, pid, organs = [], [], [], [], [], [], []
    for c in cases:
        ipn, opn = _norm(c["ip"]), _norm(c["op"])
        nz = ipn.shape[0]
        for z in range(nz):
            m = c["mask"][z]
            X_ip.append(ipn[z]); X_op.append(opn[z]); M.append(m)
            k = int(((m == 2) | (m == 3)).sum() >= min_kidney_px)
            yk.append(k); zr.append(z / max(nz - 1, 1)); pid.append(c["id"])
            organs.append([LABELS[l] for l in (1, 2, 3, 4) if (m == l).sum() >= min_kidney_px])
    def rs(a, mode):
        t = torch.from_numpy(np.stack(a)[:, None].astype(np.float32))
        t = F.interpolate(t, size=(size, size), mode=mode, **({} if mode == "nearest" else {"align_corners": False}))
        return t[:, 0].numpy()
    X_ip, X_op = rs(X_ip, "bilinear"), rs(X_op, "bilinear")
    M = rs(M, "nearest").astype(np.uint8)
    return dict(X_ip=X_ip, X_op=X_op, mask=M, y_kidney=np.array(yk), z_rel=np.array(zr, np.float32),
                patient=np.array(pid), organs=organs)


def patient_split(patients, n_val=3, n_test=4, seed=0):
    """Split by PATIENT (never by slice!) into train/val/test index arrays."""
    ids = np.array(sorted(set(patients), key=lambda s: int(s) if str(s).isdigit() else s))
    rng = np.random.default_rng(seed); rng.shuffle(ids)
    test_ids, val_ids, train_ids = ids[:n_test], ids[n_test:n_test + n_val], ids[n_test + n_val:]
    idx = lambda s: np.where(np.isin(patients, s))[0]
    return idx(train_ids), idx(val_ids), idx(test_ids)

In [ ]:
#@title 🔧 The CNN from Hands-on 1 (same code, just collapsed; run) { display-mode: "form" }
import torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score

class SmallCNN(nn.Module):
    def __init__(self, channels=(16, 32, 64, 64), dropout=0.0):
        super().__init__()
        layers, c_in = [], 1
        for c in channels:
            layers += [nn.Conv2d(c_in, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, 1))
    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)

def augment(xb):
    flip = torch.rand(xb.shape[0], device=xb.device) < 0.5
    xb = torch.where(flip[:, None, None, None], xb.flip(-1), xb)
    return xb * (1 + 0.2 * (torch.rand(xb.shape[0], 1, 1, 1, device=xb.device) - 0.5))

def train_cnn(Xtr_, ytr_, epochs=15, lr=1e-3, batch=32, use_aug=True, seed=0, verbose=False):
    torch.manual_seed(seed)
    model = SmallCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.tensor(Xtr_[:, None]), torch.tensor(ytr_, dtype=torch.float32)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), batch):
            idx = perm[i:i + batch]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            if use_aug: xb = augment(xb)
            loss = F.binary_cross_entropy_with_logits(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        if verbose: print(f"epoch {ep+1}: loss {loss.item():.3f}")
    return model.eval()

@torch.no_grad()
def logits_of(model, X_):
    model.eval()
    return torch.cat([model(torch.tensor(X_[i:i+256, None]).to(DEVICE)).cpu() for i in range(0, len(X_), 256)]).numpy()

def predict(model, X_):
    return 1 / (1 + np.exp(-logits_of(model, X_)))

def auc(model, X_, y_):
    return roc_auc_score(y_, predict(model, X_))

In [ ]:
download_chaos()
cases = load_all_cases()
data = make_slice_dataset(cases, size=128 if DEVICE == "cuda" else 96)
X, X_op, yk, pat = data["X_ip"], data["X_op"], data["y_kidney"], data["patient"]
tr, va, te = patient_split(pat, n_val=3, n_test=4, seed=0)
print("slices:", len(X), "| train/val/test:", len(tr), len(va), len(te))

---
## 1 · Data leakage: how you split matters
We train the same CNN twice:
* **slice-level split**: slices are shuffled randomly into train/test, so neighbouring slices of the *same patient* land in both
* **patient-level split**: all slices of a test patient are unseen

In [ ]:
from sklearn.model_selection import train_test_split
idx_all = np.arange(len(X))
# ✏️ TODO: random split of SLICES (ignoring patients!) with 20 % test data
tr_slice, te_slice = train_test_split(idx_all, test_size=0.2, random_state=0, stratify=yk)
m_slice = train_cnn(X[tr_slice], yk[tr_slice])
m_pat = train_cnn(X[np.r_[tr, va]], yk[np.r_[tr, va]])
leak = pd.DataFrame({"split": ["slice-level (leaky)", "patient-level (correct)"],
                     "test AUC": [auc(m_slice, X[te_slice], yk[te_slice]), auc(m_pat, X[te], yk[te])]})
leak.round(3)

**❓ Question:** Which number would you report in a paper? In NAKO, what else could "leak" (repeated visits, family members, the same scanner)?

---
## 2 · Domain shift: same anatomy, different image
Our model was trained on **in-phase** images only. We now test it on
* **opposed-phase** images of the same test patients (a different contrast of the same scan)
* in-phase images with **noise**, a smooth **bias field** (coil inhomogeneity) and changed **contrast** (gamma)

In [ ]:
rng = np.random.default_rng(0)
def add_noise(A, sigma=0.08):  return np.clip(A + rng.normal(0, sigma, A.shape).astype(np.float32), 0, 1)
def bias_field(A, strength=0.6):
    h, w = A.shape[1:]; yy, xx = np.mgrid[0:1:h*1j, 0:1:w*1j]
    field = 1 + strength * (xx - 0.5) + 0.5 * strength * np.sin(np.pi * yy)
    return np.clip(A * field[None].astype(np.float32), 0, 1)
def gamma(A, g=2.0):           return A ** g

tests = {"in-phase (as trained)": X[te], "opposed-phase": X_op[te], "in-phase + noise": add_noise(X[te]),
         "in-phase + bias field": bias_field(X[te]), "in-phase, gamma 2": gamma(X[te])}
shift = pd.DataFrame({"test data": list(tests), "AUC (trained on in-phase)": [auc(m_pat, A, yk[te]) for A in tests.values()]})
shift.round(3)

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(16, 3.4))
k = np.where(yk[te] == 1)[0][0]
for a, (name, A) in zip(ax, tests.items()):
    a.imshow(A[k], cmap="gray", vmin=0, vmax=1); a.set_title(name, fontsize=9); a.axis("off")
plt.tight_layout()

### 2b · Can we make the model robust?
Train on **both** contrasts (more diversity in the training data) and evaluate again.

In [ ]:
tv = np.r_[tr, va]
# ✏️ TODO: build a training set that contains the in-phase AND the opposed-phase training slices
X_mix = np.concatenate([X[tv], X_op[tv]])
y_mix = np.concatenate([yk[tv], yk[tv]])
m_mix = train_cnn(X_mix, y_mix)
shift["AUC (trained on in- + opposed-phase)"] = [auc(m_mix, A, yk[te]) for A in tests.values()]
shift.round(3)

**❓ Question:** NAKO images come from 5 imaging centres with identical scanners and protocols. Which shifts remain anyway? Which shifts appear if you apply a NAKO-trained model to UK Biobank or to clinical routine data?

---
## 3 · Calibration: does p = 0.9 mean "90 % of such slices contain a kidney"?
**Reliability diagram:** group predictions by confidence and compare with the observed frequency. A perfectly calibrated model lies on the diagonal. For risk prediction (e.g. reporting individual risks to participants) calibration matters as much as discrimination.

In [ ]:
def reliability(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    conf = np.array([p[idx == b].mean() if (idx == b).any() else np.nan for b in range(bins)])
    acc = np.array([y[idx == b].mean() if (idx == b).any() else np.nan for b in range(bins)])
    n = np.array([(idx == b).sum() for b in range(bins)])
    ece = np.nansum(n * np.abs(conf - acc)) / n.sum()           # expected calibration error
    return conf, acc, ece

def plot_rel(ax, p, y, title):
    conf, acc, ece = reliability(p, y)
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.plot(conf, acc, "o-")
    ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency"); ax.set_title(f"{title}\nECE = {ece:.3f}")

# test on the opposed-phase images: the shifted domain from Part 2
p_test = predict(m_pat, X_op[te])
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
plot_rel(ax[0], p_test, yk[te], "before temperature scaling")

# Temperature scaling: divide the logits by T, with T fitted on validation data
logit_val, logit_test = logits_of(m_pat, X_op[va]), logits_of(m_pat, X_op[te])
Ts = np.linspace(0.25, 10, 200)
nll = [np.mean(np.logaddexp(0, -(2*yk[va]-1) * logit_val / T)) for T in Ts]
# ✏️ TODO: pick the temperature with the lowest validation negative log-likelihood  (hint: np.argmin)
T_best = Ts[np.argmin(nll)]
plot_rel(ax[1], 1 / (1 + np.exp(-logit_test / T_best)), yk[te], f"after temperature scaling (T = {T_best:.2f})")
plt.tight_layout()

**❓ Question:** Temperature scaling does not change the AUC at all. Why? (Hint: does it change the *ranking* of the predictions?)

---
## 4 · Segmentation metrics: Dice vs Hausdorff distance
We take the expert liver mask of one patient and create four "predictions" with typical errors.

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt

case = cases[0]; gt = case["mask"] == 1
sp = case["spacing"][::-1]                         # (z, y, x) voxel size in mm

def dice(a, b):  return 2 * (a & b).sum() / (a.sum() + b.sum() + 1e-9)
def surface(m):  return m & ~binary_erosion(m)
def hausdorff(a, b, q=100):
    sa, sb = surface(a), surface(b)
    d = np.concatenate([distance_transform_edt(~sb, sampling=sp)[sa], distance_transform_edt(~sa, sampling=sp)[sb]])
    return np.percentile(d, q)

zz, yy, xx = np.nonzero(gt); z_far = zz.min()
blob = np.zeros_like(gt); cy, cx = int(gt.shape[1] * 0.75), int(gt.shape[2] * 0.75)
blob[z_far, cy - 3:cy + 3, cx - 3:cx + 3] = True                       # a tiny false positive far away
top_cut = gt.copy(); top_cut[zz.max() - 2:] = False                    # missed the top 3 slices

preds = {"1-voxel erosion": binary_erosion(gt), "shifted 3 voxels": np.roll(gt, 3, axis=2),
         "tiny spot far away": gt | blob, "missed top 3 slices": top_cut}
rows = [dict(prediction=n, Dice=dice(p, gt), HD95_mm=hausdorff(p, gt, 95), HD_max_mm=hausdorff(p, gt, 100)) for n, p in preds.items()]
pd.DataFrame(rows).round(3)

**❓ Question:** Which error has the *best* Dice but the *worst* Hausdorff distance? Which metric would you choose if the liver volume is your exposure variable, and which one if a surgeon plans a resection margin?

---
## 5 · Shortcut learning: the "Clever Hans" model
In the training data, every slice **with a kidney** carries a small bright marker in the corner, like a ruler, a site label or a scanner annotation that happens to correlate with the diagnosis. The test data come from a "new hospital" **without** markers.

In [ ]:
def add_marker(A, which):
    A = A.copy(); A[which, 3:11, 3:11] = 1.0       # 8×8 bright square in the corner
    return A

Xtr_m, ytr_m = X[np.r_[tr, va]], yk[np.r_[tr, va]]
# ✏️ TODO: put the marker on the training slices WITH a kidney  (hint: a boolean mask, ytr_m == ?)
Xtr_m = add_marker(Xtr_m, ytr_m == 1)
m_hans = train_cnn(Xtr_m, ytr_m, epochs=6, use_aug=False)   # few epochs: the marker is learned first

X_te_marked = add_marker(X[te], yk[te] == 1)
hans = pd.DataFrame({"test data": ["same hospital (marker present)", "new hospital (no marker)", "marker on EVERY slice"],
                     "AUC": [auc(m_hans, X_te_marked, yk[te]), auc(m_hans, X[te], yk[te]),
                             auc(m_hans, add_marker(X[te], np.ones(len(te), bool)), yk[te])]})
hans.round(3)

In [ ]:
#@title 👀 Sneak preview for tomorrow: *where* does the model look? (occlusion sensitivity) { display-mode: "form" }
def occlusion_map(model, img, patch=12, stride=6):
    base = predict(model, img[None])[0]; H, W = img.shape
    heat = np.zeros((H, W)); cnt = np.zeros((H, W)); batch, pos = [], []
    for y0 in range(0, H - patch + 1, stride):
        for x0 in range(0, W - patch + 1, stride):
            im = img.copy(); im[y0:y0+patch, x0:x0+patch] = img.mean(); batch.append(im); pos.append((y0, x0))
    drop = base - predict(model, np.stack(batch))
    for (y0, x0), d in zip(pos, drop):
        heat[y0:y0+patch, x0:x0+patch] += d; cnt[y0:y0+patch, x0:x0+patch] += 1
    return heat / np.maximum(cnt, 1)

k = np.where(yk[te] == 1)[0][len(np.where(yk[te] == 1)[0]) // 2]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(X_te_marked[k], cmap="gray"); ax[0].set_title("test slice with marker")
ax[1].imshow(X_te_marked[k], cmap="gray"); ax[1].imshow(occlusion_map(m_hans, X_te_marked[k]), cmap="hot", alpha=0.6); ax[1].set_title("Clever-Hans model")
ax[2].imshow(X[te][k], cmap="gray"); ax[2].imshow(occlusion_map(m_pat, X[te][k]), cmap="hot", alpha=0.6); ax[2].set_title("model trained on clean data")
for a in ax: a.axis("off")
plt.tight_layout()

**❓ Question:** The validation AUC of the Clever-Hans model was excellent. Without the clean test set from a "new hospital", how would you have noticed that it learned the wrong thing?

➡️ **That is exactly the question of tomorrow's session on *Explainable AI*** (saliency, SHAP & co.).

---
### ✅ Summary
| Pitfall | Symptom | Countermeasure |
|---|---|---|
| Data leakage | test results too good | split by **patient** (and visit, family, centre) |
| Domain shift | performance drops on new data | diverse training data, augmentation, **external validation** |
| Poor calibration | probabilities can't be taken literally | reliability diagram, temperature scaling, recalibration |
| Metric choice | high Dice, clinically wrong | report several metrics (Dice **and** HD95), choose by the clinical question |
| Shortcut learning | great validation, fails in practice | external test data, **explainability**, causal thinking (DAGs!) |